# 02 — Обучение моделей каскада

Исполнимый pipeline: regex coverage → embeddings → XGBoost per-attribute → калибровка thresholds → Bayes fit → category router + Mahalanobis OOD.

**Default HEAVY=False:** читаем `models/` если есть, иначе warning. Полный train на VM (см. CLAUDE.md Remote VM).

Артефакт-граница (см. spec §6.2 / §8): этот ноутбук читает `datasets/processed/`, пишет в `models/` и `datasets/processed/{cat}_v4_embeddings.npy`.

**Замечание про `models/`:** в commit `3a7bc26` все silver-trained `.pkl` были удалены — затем восстановлены частично (v4 LLM-relabel gold). Если каких-то моделей нет — ячейки печатают warning и не падают.

In [1]:
%env OMP_NUM_THREADS=1

import sys
from pathlib import Path

assert sys.version_info[:2] in [(3, 12), (3, 14)], (
    f"Untested Python {sys.version_info[:2]}; pickle compat между 3.12 (VM) и 3.14 (local) для XGBoost не гарантирована"
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / 'datasets' / 'processed'
MODELS = PROJECT_ROOT / 'models'

CATEGORIES = ['pasta_v4', 'chocolate_v4', 'cheeses_v4']

HEAVY = {
    'embed': False,  # ~30 минут SentenceTransformer encode на CPU
    'train': False,  # XGBoost per-attr + Bayes + router
}

import subprocess
import pandas as pd
import numpy as np
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'models/ exists: {MODELS.exists()}, .pkl files: {len(list(MODELS.glob("*.pkl"))) if MODELS.exists() else 0}')


env: OMP_NUM_THREADS=1


PROJECT_ROOT = /Users/miafrolov/Desktop/stuff/ai_attributes
models/ exists: True, .pkl files: 210


## 1. Layer 1 — Regex coverage

Layer 1 не требует обучения: чистая ручная инженерия паттернов (`src/pipeline/regex/extractor.py`). Здесь проверяем покрытие на silver: доля строк, где regex успел извлечь значение, — потолок "бесплатных" предсказаний перед ML-слоем.

In [2]:
try:
    from src.pipeline.regex.extractor import RegexExtractor
except ImportError as e:
    print(f'Regex extractor unavailable: {e}')
    RegexExtractor = None

coverage = {}
if RegexExtractor is not None:
    ext = RegexExtractor()
    for cat in ['pasta', 'chocolate', 'cheeses']:
        silver = PROCESSED / f'{cat}_stratified_silver_standard.parquet'
        if not silver.exists():
            print(f'{cat}: silver missing ({silver.name})')
            continue
        try:
            df = pd.read_parquet(silver).head(500)
        except Exception as e:
            print(f'{cat}: read failed — {e}')
            continue
        results = []
        for _, r in df.iterrows():
            try:
                out = ext.extract_all(
                    product_name=str(r.get('product_name', '') or ''),
                    quantity=str(r.get('quantity', '') or ''),
                    category=cat,
                    brands=str(r.get('brands', '') or ''),
                    ingredients_text=str(r.get('ingredients_text', '') or ''),
                )
                # extract_all возвращает dict[attr -> ExtractionResult]; .value — None если не сработал
                results.append({k: v.value for k, v in out.items()})
            except Exception as e:
                results.append({})
        if results:
            cov = pd.DataFrame(results).notna().mean().mul(100).round(1)
            coverage[cat] = cov

if coverage:
    cov_df = pd.DataFrame(coverage).fillna('—')
    print('Layer 1 coverage % (доля непустых regex-извлечений на первых 500 строках silver):')
    try:
        from IPython.display import display
        display(cov_df)
    except Exception:
        print(cov_df)
else:
    print('coverage не посчитан (нет silver или extractor недоступен)')


Layer 1 coverage % (доля непустых regex-извлечений на первых 500 строках silver):


,pasta,chocolate,cheeses
chocolate_type,—,61.8,—
cocoa_percentage,—,69.4,—
contains_nuts,—,38.8,—
cooking_time,0.0,0.0,0.0
fat_content,27.8,16.8,8.4
grain_type,87.0,—,—
is_pdo,—,—,5.8
is_ultra_processed,—,—,0.4
measure,97.6,98.6,98.8
milk_source,—,—,8.6


## 2. Layer 2 — Embeddings (MPNet 768d)

`paraphrase-multilingual-mpnet-base-v2`, ~30 минут на CPU. Кэш в `datasets/processed/{cat}_v4_embeddings.npy`. При `HEAVY['embed']=False` переиспользуется кэш или печатается warning.

In [3]:
for cat in CATEGORIES:
    emb_path = PROCESSED / f'{cat}_embeddings.npy'
    # gold v4 wide parquet — основной источник текстов для encode
    cat_short = cat[:-3]  # 'pasta_v4' -> 'pasta'
    gold_path = PROCESSED / f'{cat_short}_gold_v4_wide.parquet'
    if HEAVY['embed']:
        try:
            from sentence_transformers import SentenceTransformer  # lazy
            if not gold_path.exists():
                print(f'{cat}: gold v4 wide missing ({gold_path.name}); see 01_dataset.ipynb')
                continue
            model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
            gold = pd.read_parquet(gold_path)
            texts = (
                gold['product_name'].fillna('') + ' ' +
                gold.get('brands', pd.Series(['']*len(gold))).fillna('') + ' ' +
                gold.get('ingredients_text', pd.Series(['']*len(gold))).fillna('')
            ).tolist()
            emb = model.encode(texts, show_progress_bar=True, batch_size=64)
            np.save(emb_path, emb)
            print(f'{cat}: encoded {emb.shape} -> {emb_path.name}')
        except Exception as e:
            print(f'{cat}: encode failed — {e}')
    elif emb_path.exists():
        emb = np.load(emb_path)
        print(f'{cat}: cached {emb.shape} ({emb_path.name})')
    else:
        print(f'{cat}: embeddings отсутствуют ({emb_path.name}); set HEAVY["embed"]=True для encode на VM')


pasta_v4: embeddings отсутствуют (pasta_v4_embeddings.npy); set HEAVY["embed"]=True для encode на VM
chocolate_v4: embeddings отсутствуют (chocolate_v4_embeddings.npy); set HEAVY["embed"]=True для encode на VM
cheeses_v4: embeddings отсутствуют (cheeses_v4_embeddings.npy); set HEAVY["embed"]=True для encode на VM


## 3. Layer 2 — XGBoost per-attribute fit

Обёртка вокруг CLI `src.pipeline.ml.train` (см. spec §6.2: модули остаются CLI, ноутбук вызывает их как subprocess до рефакторинга в library API). При `HEAVY['train']=False` только проверяем наличие моделей в `models/`.

In [4]:
ML_CATEGORIES = ['pasta_stratified', 'chocolate_stratified', 'cheeses_stratified']

for cat in ML_CATEGORIES:
    if HEAVY['train']:
        cmd = ['python', '-m', 'src.pipeline.ml.train', '--category', cat]
        print(f'>> {" ".join(cmd)}')
        try:
            r = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True,
                               text=True, timeout=3600)
            if r.stdout:
                print(r.stdout[-1000:])
            if r.returncode != 0:
                print(f'STDERR (last 500): {r.stderr[-500:]}')
        except subprocess.TimeoutExpired:
            print(f'{cat}: timeout (>1h); запускай на VM с background nohup')
        except Exception as e:
            print(f'{cat}: subprocess failed — {e}')
    pkls = list(MODELS.glob(f'{cat}_*_xgb.pkl'))
    v4_short = cat.replace('_stratified', '_v4')
    pkls_v4 = list(MODELS.glob(f'{v4_short}_*_xgb.pkl'))
    print(f'{cat}: {len(pkls)} silver-XGB; {len(pkls_v4)} v4-XGB ({v4_short}_*_xgb.pkl)')

if not HEAVY['train']:
    print('\n[info] HEAVY["train"]=False — only inventory shown. '
          'Set HEAVY["train"]=True (рекомендуется на VM) для полного fit.')


pasta_stratified: 0 silver-XGB; 28 v4-XGB (pasta_v4_*_xgb.pkl)
chocolate_stratified: 0 silver-XGB; 20 v4-XGB (chocolate_v4_*_xgb.pkl)
cheeses_stratified: 0 silver-XGB; 28 v4-XGB (cheeses_v4_*_xgb.pkl)

[info] HEAVY["train"]=False — only inventory shown. Set HEAVY["train"]=True (рекомендуется на VM) для полного fit.


## 4. Калибровка confidence thresholds

Per-attribute изотоническая калибровка → `{cat}_{attr}_calibration.json`; пороги — `{cat}_thresholds.pkl`. Артефакты появляются автоматически как часть `src.pipeline.ml.train` (см. секцию 3). Здесь только smoke-инвентаризация.

In [5]:
import json

rows = []
for cat in ['pasta_stratified', 'chocolate_stratified', 'cheeses_stratified']:
    # ищем и silver-, и v4- варианты
    variants = [cat, cat.replace('_stratified', '_v4'), cat.replace('_stratified', '_v4_mpnet')]
    for v in variants:
        thr = MODELS / f'{v}_thresholds.pkl'
        calibs = list(MODELS.glob(f'{v}_*_calibration.json'))
        if thr.exists() or calibs:
            rows.append({
                'prefix': v,
                'thresholds.pkl': 'OK' if thr.exists() else 'MISSING',
                'calibration_jsons': len(calibs),
            })
if rows:
    df = pd.DataFrame(rows)
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df)
else:
    print('Калибровочные артефакты отсутствуют. '
          'Запусти HEAVY["train"]=True (рекомендуется на VM) — '
          'src.pipeline.ml.train сохранит thresholds + per-attr calibration.json.')


,prefix,thresholds.pkl,calibration_jsons
0,pasta_v4,OK,28
1,pasta_v4_mpnet,OK,21
2,chocolate_v4,OK,20
3,chocolate_v4_mpnet,OK,15
4,cheeses_v4,OK,28
5,cheeses_v4_mpnet,OK,21


## 5. Layer 3 — Bayesian network (Hill Climb + BIC)

`src.pipeline.bayes.train` — pgmpy 1.1.2 DiscreteBayesianNetwork, структура Hill Climb + BIC, CPD через BayesianEstimator. Артефакт: `{cat}_stratified_bayesian.pkl`.

In [6]:
for cat in ['pasta_stratified', 'chocolate_stratified', 'cheeses_stratified']:
    bayes_pkl = MODELS / f'{cat}_bayesian.pkl'
    if HEAVY['train']:
        cmd = ['python', '-m', 'src.pipeline.bayes.train', '--category', cat]
        print(f'>> {" ".join(cmd)}')
        try:
            r = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True,
                               text=True, timeout=1800)
            if r.stdout:
                print(r.stdout[-500:])
            if r.returncode != 0:
                print(f'STDERR: {r.stderr[-500:]}')
        except subprocess.TimeoutExpired:
            print(f'{cat}: timeout (>30min)')
        except Exception as e:
            print(f'{cat}: subprocess failed — {e}')
    print(f'{cat} bayesian.pkl: {"OK" if bayes_pkl.exists() else "MISSING"}')

if not HEAVY['train']:
    bayes_count = len(list(MODELS.glob('*_bayesian.pkl')))
    if bayes_count == 0:
        print('\n[warn] Ни одного *_bayesian.pkl в models/. '
              'Запусти HEAVY["train"]=True (рекомендуется на VM) — Bayes train ~5-15 мин на категорию.')


pasta_stratified bayesian.pkl: MISSING
chocolate_stratified bayesian.pkl: MISSING
cheeses_stratified bayesian.pkl: MISSING

[warn] Ни одного *_bayesian.pkl в models/. Запусти HEAVY["train"]=True (рекомендуется на VM) — Bayes train ~5-15 мин на категорию.


## 6. Layer 0 — Category router + Mahalanobis OOD

Pre-cascade XGBoost-классификатор (7 known categories) + порог-OOD по Mahalanobis distance. Текущая prod-версия: `category_router_v5.pkl` (см. MEMORY.md).

In [7]:
router_pkls = sorted(MODELS.glob('category_router_*'))
mahal_jsons = sorted(MODELS.glob('*mahalanobis*'))

if HEAVY['train']:
    cmd = ['python', '-m', 'src.pipeline.category_router.train']
    print(f'>> {" ".join(cmd)}')
    try:
        r = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True,
                           text=True, timeout=1800)
        if r.stdout:
            print(r.stdout[-500:])
        if r.returncode != 0:
            print(f'STDERR: {r.stderr[-500:]}')
    except subprocess.TimeoutExpired:
        print('router: timeout (>30min)')
    except Exception as e:
        print(f'router: subprocess failed — {e}')
    # Mahalanobis fit — отдельный шаг (см. src/pipeline/category_router/fit_mahalanobis.py)
    cmd2 = ['python', '-m', 'src.pipeline.category_router.fit_mahalanobis']
    print(f'>> {" ".join(cmd2)}')
    try:
        r2 = subprocess.run(cmd2, cwd=str(PROJECT_ROOT), capture_output=True,
                            text=True, timeout=900)
        if r2.stdout:
            print(r2.stdout[-500:])
        if r2.returncode != 0:
            print(f'STDERR: {r2.stderr[-500:]}')
    except Exception as e:
        print(f'mahalanobis: {e}')

print(f'\nrouter artifacts ({len(router_pkls)}): {[p.name for p in router_pkls[:5]]}')
print(f'mahalanobis artifacts ({len(mahal_jsons)}): {[p.name for p in mahal_jsons[:5]]}')

if not router_pkls and not HEAVY['train']:
    print('\n[warn] Router отсутствует — set HEAVY["train"]=True')
if not mahal_jsons and not HEAVY['train']:
    print('[warn] Mahalanobis OOD-параметры отсутствуют — fit_mahalanobis.py обычно отдельный шаг')



router artifacts (3): ['category_router_v5.pkl', 'category_router_v5_le.pkl', 'category_router_v5_meta.json']
mahalanobis artifacts (0): []
[warn] Mahalanobis OOD-параметры отсутствуют — fit_mahalanobis.py обычно отдельный шаг


---

## Готово

При `HEAVY={'embed': False, 'train': False}` это smoke-инвентаризация (показывает что есть в `models/`). Для полного train: `HEAVY={'embed': True, 'train': True}` и запуск на VM (см. CLAUDE.md Remote VM, rsync local↔VM, nohup background).